# H-Beauty: free test of an AI teacher for the phone model

Runs **FLUX 3 Image Klein** on Kaggle's free GPU on the 3 hardest looks × 4 synthetic faces, and (optional, ~$0.50)
the same 12 with **Nano Banana 2.1** to compare. Nothing here goes into the app; it only decides which teacher to use.

**Settings (right panel, once):**
1. *Session options* → **Accelerator: GPU T4 x2**, **Internet: on** (needs a phone-verified Kaggle account).
2. *Add Input* → your dataset made from `hb-gpu-kit.zip`.
3. *Add-ons → Secrets*: **HF_TOKEN** (your Hugging Face Read token; accept the terms on the
   `black-forest-labs/flux-3-image-klein` page first) and, optional, **GEMINI_API_KEY** (Google AI Studio).
   Switch both on for this notebook.

Then **Save Version → Save & Run All**. It runs in the background (~40–60 min); you can close the page.
When it's done, open the version → *Output* → download **test-results.zip** and send it in the chat.

In [ ]:
# 1. Find the kit, read the secrets, install (2-3 min)
import glob, os, shutil
hits = glob.glob('/kaggle/input/**/scripts/gpu/free_test.py', recursive=True)
assert hits, 'Add your hb-gpu-kit dataset: right panel -> Add Input'
shutil.copytree(hits[0].split('/scripts/gpu/')[0], '/kaggle/working/kit', dirs_exist_ok=True)

from kaggle_secrets import UserSecretsClient
def secret(name):
    try: return UserSecretsClient().get_secret(name) or ''
    except Exception: return ''
os.environ['HF_TOKEN'] = secret('HF_TOKEN')
os.environ['GEMINI_API_KEY'] = secret('GEMINI_API_KEY')
assert os.environ['HF_TOKEN'], 'Add the HF_TOKEN secret (Add-ons -> Secrets) and switch it on for this notebook'
print('Nano Banana comparison:', 'on' if os.environ['GEMINI_API_KEY'] else 'off (no GEMINI_API_KEY)')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
!pip install -q -U diffusers transformers accelerate huggingface_hub hf_transfer google-genai mediapipe opencv-python-headless
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!free -g | head -2; df -h / /kaggle/working | tail -2

In [ ]:
# 2. The test (~40-60 min on the free GPU): each finished image prints a line
!cd /kaggle/working && python kit/scripts/gpu/free_test.py 2>&1 | grep -v -E 'W0000|I0000|XNNPACK|feedback tensors|Exception ignored|TypeError: .NoneType'

In [ ]:
# 3. The sheets (also in test-results.zip under Output)
from IPython.display import Image, display
for f in sorted(glob.glob('/kaggle/working/test-results/sheet_*.jpg')):
    print(f.split('/')[-1]); display(Image(filename=f, width=1200))